# 03: Forgetting diagnostics from a saved checkpoint

Start in a **fresh Colab GPU runtime** and run these cells in order. The previous experiment's model is on Google Drive; nothing in this notebook trains a model.

Our next comparison is **E10**: forgetting learning rate `0.00001`, gamma `0.00001`, ten updates. E09 used the same forgetting learning rate with gamma `0.0001`. Both must start from the same checkpoint.

Read the [experiment log](https://github.com/sumitasthana/hypernetworks/blob/main/docs/EXPERIMENT_LOG.md) for all reported results. E10 has no reported result yet. Stop after this one comparison and share its table.

## 1. Get the updated repository

This cell installs the local package without replacing Colab's PyTorch installation. It checks for an already imported old package and refuses to overwrite changes in an existing clone. A checkout already on another branch is also left alone.


In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys

# A fresh runtime avoids mixing old imported classes with the new helper.
if "uncle" in sys.modules:
    raise RuntimeError("Restart the runtime, then run this notebook from the top.")

REPO = Path("/content/uncle-diagnostics")
REPO_URL = "https://github.com/sumitasthana/hypernetworks"

# Clone into local runtime storage. Preserve any existing uncommitted work.
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(REPO)], check=True)
else:
    def git_output(*args):
        return subprocess.check_output(["git", "-C", str(REPO), *args], text=True).strip()

    if git_output("remote", "get-url", "origin").removesuffix(".git") != REPO_URL:
        raise RuntimeError("This directory contains a different repository.")
    if git_output("status", "--porcelain"):
        raise RuntimeError("The clone has local changes. Preserve them before updating it.")
    if git_output("branch", "--show-current") != "main":
        raise RuntimeError("The clone is not on main. Use a fresh clone directory.")
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)

# Colab supplies torch, torchvision, NumPy, and Pillow. Keep its GPU builds.
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", str(REPO)
], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
importlib.invalidate_caches()
print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True).strip())


## 2. Mount Drive and check the saved model

Images stay on runtime-local disk for faster access. The original checkpoint and new reports stay on Drive so they survive a session ending.

If the checkpoint is missing, stop and check the path or Drive account. Do not silently replace it with a newly trained model: E09 and E10 need a shared starting point.


In [ ]:
from google.colab import drive
import torch
import uncle
from uncle import diagnose_forgetting

drive.mount("/content/drive")
DATA = Path("/content/data/tiny-imagenet-200")
CHECKPOINT = Path(
    "/content/drive/MyDrive/uncle/E08_forgetting_trace/before_forgetting.pt"
)
OUTPUT = CHECKPOINT.parent / "diagnostics"

# Fail early if the intended GPU or persistent model is unavailable.
assert torch.cuda.is_available(), "Select a GPU runtime before proceeding."
assert CHECKPOINT.is_file(), f"Checkpoint not found: {CHECKPOINT}"
assert Path(uncle.__file__).resolve().parent.parent == REPO.resolve(), (
    "An unexpected copy of uncle was imported. Restart the runtime."
)
print("GPU:", torch.cuda.get_device_name(0))
print("Package:", uncle.__file__)
print("Checkpoint:", CHECKPOINT)
print("New reports:", OUTPUT)


## 3. Inspect the checkpoint without training

Expected saved accuracies are task 3 = **26.0%**, task 0 = **44.6%**. This cell reads recorded metadata; the diagnostic will also evaluate the restored model itself.

The checkpoint loader handles the earlier RNG tensor issue. Do not add manual type conversions or skip random-state entries.


In [ ]:
from uncle import checkpoint as checkpointing

# Read the checkpoint on CPU, inspect its metadata, then release the payload.
# No model is trained and no checkpoint is modified by this cell.
saved = checkpointing.load(CHECKPOINT)
assert saved is not None
assert [(r["action"], r["task"]) for r in saved["history"]] == [
    ("learn", "3"), ("learn", "0")
]
assert saved["forgotten"] == [], "This must be a pre-forgetting checkpoint."
expected_start = {"3": 26.0, "0": 44.6}
assert all(abs(saved["previous"][t] - a) < 1e-6 for t, a in expected_start.items()), (
    "This is a different starting checkpoint. Review it before comparing with E09."
)
for name in ("backbone", "epochs", "learning_rate", "seed", "chunks"):
    print(name, saved["config"][name])
print("Saved accuracies:", saved["previous"])
del saved


## 4. Run E10 with one library call

**Only gamma changes relative to E09.** Keep the forgetting learning rate and update count fixed. The helper restores the checkpoint on every call, uses one optimizer and one frozen reference, and evaluates at step zero and after every update.

The dataset may need downloading again in this new runtime. `download=True` permits that; it does not request retraining. Evaluation uses the full validation splits.

If interrupted, rerun this cell. It restarts from the original checkpoint and creates a new report rather than continuing an incomplete forgetting trace.


In [ ]:
# E10: a tenfold smaller noise coefficient than E09.
# Change future experiment settings here, never inside a copied training loop.
FORGETTING_LR = 1e-5
GAMMA = 1e-5
STEPS = 10

report = diagnose_forgetting(
    checkpoint=CHECKPOINT,      # Restored afresh; never overwritten.
    task="3",                  # Forget this task; protect the other seen task.
    root=DATA,
    download=True,             # Prepare local images if the runtime is new.
    forgetting_lr=FORGETTING_LR,
    gamma=GAMMA,
    steps=STEPS,                # Exactly ten continuous Adam updates.
    output=OUTPUT,             # Save per-step measurements persistently on Drive.
)
print("Saved report:", report["report_path"])


## 5. Apply the existing screen and stop

Both starting accuracies must be at least 25%. A candidate step must have task 3 at or below 12%, and task 0's absolute drift below five percentage points. These are diagnostic screens, not proof of deletion.

Losses printed by the helper are measured **before** each update. Accuracies are measured **after** it. A lower loss after changing gamma is not itself evidence of improvement.


In [ ]:
import json

# Check actual restored predictions before interpreting the comparison.
assert report["status"] == "complete", "The trace is incomplete."
assert report["initial_matches_checkpoint"], (
    "Initial predictions differ from the saved record. Share the report before proceeding."
)
initial = report["trace"][0]["accuracies"]
valid = all(initial[t] >= 25.0 for t in ("3", "0"))
passing_steps = []

print("step  task3  task0  task0_drift  passes_screen")
for row in report["trace"]:
    a = row["accuracies"]
    drift = abs(a["0"] - initial["0"])
    passed = valid and row["step"] > 0 and a["3"] <= 12.0 and drift < 5.0
    if passed:
        passing_steps.append(row["step"])
    print(f"{row['step']:4d}  {a['3']:5.1f}  {a['0']:5.1f}  {drift:11.1f}  {passed}")

# Save the interpretation separately; retain the original measurement report.
screen = {
    "experiment": "E10",
    "report_path": report["report_path"],
    "thresholds": {"min_before": 25.0, "target_at_most": 12.0, "drift_under": 5.0},
    "valid_start": valid,
    "passing_steps": passing_steps,
}
screen_path = Path(report["report_path"]).with_suffix(".screen.json")
screen_path.write_text(json.dumps(screen, indent=2) + "\n", encoding="utf-8")
print("Passing steps:", passing_steps)
print("Screening summary:", screen_path)
print("STOP: share the table and report before selecting another experiment.")


## What to share

Send the helper's full per-step table, the screening table, GPU name, and the JSON report path. If a cell fails, share its traceback instead of bypassing its checks.

This notebook is a fresh interface to the existing saved model. It does not reset the research history, rerun the previous experiments, or establish an E10 result until its cells are executed.
